<div style="background-color: #ffffff; color: #000000; padding: 30px;">
<img src="../media/images/kisz_logo.png" width="192" height="69" align="right" style="margin-right: 50px; margin-bottom: 50px;">
<h1>Time Series Analysis and Forecasting</h1>
</div>

<div style="background-color: #f6a800; color: #ffffff; padding: 10px;">
<h2>Part C: Machine Learning Approaches</h2>
<h2>Notebook C01: Feature Engineering for Forecasting</h2>
</div>

Machine learning models know nothing about time. A gradient boosting model sees a table of rows and has
no idea that one row comes after another, or that the order matters at all. Everything a statistical model
gets for free from its structure, an ML model has to be handed as a column.

That is what feature engineering is here: turning a sequence into a table, without accidentally putting
the future into it. The second half of that sentence is the hard part, and it gets a section of its own.

---

**Contents**

1. [Imports and Data Loading](#1.-Imports-and-Data-Loading)
2. [Forecasting as Supervised Learning](#2.-Forecasting-as-Supervised-Learning)
3. [Lag Features](#3.-Lag-Features)
4. [Rolling and Expanding Statistics](#4.-Rolling-and-Expanding-Statistics)
5. [Leakage: The Mistake That Matters Most](#5.-Leakage:-The-Mistake-That-Matters-Most)
6. [Calendar and Fourier Features](#6.-Calendar-and-Fourier-Features)
7. [Preparing the Target](#7.-Preparing-the-Target)
8. [Which Features Matter](#8.-Which-Features-Matter)

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="1.-Imports-and-Data-Loading">1. Imports and Data Loading</h3>
</div>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold, TimeSeriesSplit, cross_val_score
from statsmodels.tsa.stattools import adfuller

import nb_config

sns.set_theme(style="whitegrid")

We use the Rossmann daily sales data, the same store as Notebooks
[A04](./A04_Handling_outliers.ipynb) and [B02](./B02_ARIMA_models.ipynb). It suits this notebook because
it has everything feature engineering deals with: a weekly cycle, a yearly one, and columns that are known
in advance (promotions, holidays, opening days).

If you have not prepared it yet, run Notebook [F01c](./F01c_Preparing_external_datasets.ipynb).

In [ ]:
sales = pd.read_csv(nb_config.ROSSMANN_TRAIN_PATH, parse_dates=["Date"], low_memory=False)

store = (
    sales[sales["Store"] == 1]
    .set_index("Date")
    .sort_index()
    .asfreq("D")                # daily, including the days the shop was shut
)

target = store["Sales"].astype(float)

print(f"{len(store)} days, {store.index.min().date()} to {store.index.max().date()}")
store[["Sales", "Open", "Promo", "SchoolHoliday"]].head()

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="2.-Forecasting-as-Supervised-Learning">2. Forecasting as Supervised Learning</h3>
</div>

A statistical model is handed a series and works out the dependence itself. A supervised learning model
needs a matrix $X$ of features and a vector $y$ of targets, where **each row is one prediction problem**:
everything you would know at the moment of predicting, alongside the thing you are trying to predict.

For forecasting, the row for day $t$ holds the target $y_t$ and features built only from information
available before $t$. The whole craft is in that last clause.

Three consequences follow immediately, and all three are ways people get this wrong:

**The rows are not independent.** Consecutive days are correlated, which breaks the assumption behind
most of what scikit-learn does. The models still work; the *evaluation* is what breaks.

**The order is information.** Shuffling rows destroys it. `train_test_split(shuffle=True)`, the default,
trains on next year and tests on last year.

**The future must never appear in a feature.** Not directly, and not through a statistic computed over a
period that includes it. This is called leakage, and it is the subject of section 5.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="3.-Lag-Features">3. Lag Features</h3>
</div>

The most basic feature is the target itself, moved forward in time. A **lag** of 1 gives yesterday's
sales as a predictor of today's; a lag of 7 gives the same weekday last week.

`shift(k)` moves values forward by $k$ rows, so `shift(1)` puts yesterday's value on today's row. Which
lags to include follows from what you learned in Part A: the ACF and PACF plots of Notebook
[A02](./A02_Basic_plotting.ipynb) tell you which past values carry information.

In [ ]:
def add_lags(features, target, lags):
    """Value of the target `lag` days ago."""
    for lag in lags:
        features[f"lag_{lag}"] = target.shift(lag)
    return features


demo = pd.DataFrame({"sales": target})
add_lags(demo, target, [1, 2, 7])

demo.head(9)

Read the first rows carefully, because they contain the whole idea. On any given row, `lag_1` holds the
previous row's `sales`. The model sees only that column, never the `sales` value on its own row.

The missing values at the top are unavoidable: the first day has no yesterday. Every lag you add costs you
rows at the start of the series, and a lag of 28 costs 28 of them.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="4.-Rolling-and-Expanding-Statistics">4. Rolling and Expanding Statistics</h3>
</div>

A single lag is a single noisy observation. **Rolling statistics** summarise a stretch of recent history
instead: the mean of the last week smooths out day-to-day noise, and the standard deviation of the last
week says how settled things have been.

The mechanics matter here. `rolling(7).mean()` on day $t$ averages days $t-6$ through $t$ **inclusive**,
so it contains the value you are trying to predict. The fix is to shift first and roll afterwards:

```python
target.shift(1).rolling(7).mean()
```

which averages days $t-7$ through $t-1$. Getting this the wrong way round is the single most common bug in
ML forecasting, and section 5 shows what it costs.

In [ ]:
def add_rolling(features, target, windows, shift=1):
    """Rolling summaries of the `shift` days before each row."""
    history = target.shift(shift)
    for window in windows:
        features[f"roll_mean_{window}"] = history.rolling(window).mean()
        features[f"roll_std_{window}"] = history.rolling(window).std()
        features[f"roll_max_{window}"] = history.rolling(window).max()
    return features


def add_expanding(features, target, shift=1):
    """Everything known so far, rather than a fixed window."""
    history = target.shift(shift)
    features["expanding_mean"] = history.expanding(min_periods=28).mean()
    return features


demo = pd.DataFrame({"sales": target})
add_rolling(demo, target, [7])
add_expanding(demo, target)

demo.loc["2013-02-01":"2013-02-08"].round(1)

An **expanding** window uses all history rather than a fixed span, so it behaves like a mean that keeps
being refined. It is more stable than a rolling mean and slower to react, which makes it useful for
capturing a level that drifts over years rather than weeks.

Note that the rolling features are also a way to smuggle exogenous information in: rolling statistics of
*another* series, such as a competitor's prices or last week's weather, are built exactly the same way.

**Exercise.** Add a **seasonal rolling** feature: the mean of the same weekday over the previous four weeks (`target.shift(7).rolling(4)` applied on a weekday basis, or `target.shift(7).rolling(28).mean()` as an approximation). Does it correlate more strongly with the target than the plain 7-day mean?

In [ ]:
# Your solution here


---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="5.-Leakage:-The-Mistake-That-Matters-Most">5. Leakage: The Mistake That Matters Most</h3>
</div>

**Leakage** is when a feature contains information that would not have been available at prediction time.
The model learns to use it, the offline scores look excellent, and the thing fails on deployment. It is
the defining failure of machine learning on time series, and it is almost always an accident.

The version below is a natural-looking mistake: a **centred** rolling mean. `center=True` is a perfectly
sensible option for smoothing a series you are plotting, and it is fatal in a feature matrix, because it
averages days on both sides of each row.

In [ ]:
def build_features(target, store, leaky=False):
    """The same feature set, built correctly or with a centred rolling window."""
    features = pd.DataFrame(index=target.index)

    add_lags(features, target, [1, 7, 14])

    if leaky:
        # WRONG: centred windows look both ways, so each row sees its own future
        for window in (3, 7, 28):
            features[f"roll_mean_{window}"] = target.rolling(window, center=True, min_periods=1).mean()
    else:
        add_rolling(features, target, [7, 28])

    features["day_of_week"] = target.index.dayofweek
    features["month"] = target.index.month
    features["open"] = store["Open"]
    features["promo"] = store["Promo"]

    return features


def evaluate(features, target, holdout_days=90):
    """Score the same model three ways."""
    complete = features.notna().all(axis=1)
    X, y = features[complete], target[complete]
    split = len(X) - holdout_days

    model = HistGradientBoostingRegressor(random_state=0)

    shuffled = -cross_val_score(
        model, X, y, cv=KFold(5, shuffle=True, random_state=0),
        scoring="neg_mean_absolute_error",
    ).mean()
    ordered = -cross_val_score(
        model, X, y, cv=TimeSeriesSplit(5), scoring="neg_mean_absolute_error"
    ).mean()

    model.fit(X.iloc[:split], y.iloc[:split])
    holdout = mean_absolute_error(y.iloc[split:], model.predict(X.iloc[split:]))

    return {"Shuffled K-fold CV": shuffled, "TimeSeriesSplit CV": ordered, "Future holdout": holdout}


scores = pd.DataFrame({
    "Correct features": evaluate(build_features(target, store), target),
    "Leaky features": evaluate(build_features(target, store, leaky=True), target),
})

scores.round(1)

The leaky feature set wins on every measure, and on the future holdout it looks **26% better**: 231
against 314. If you built this, cross-validated it, and reported it, nobody reviewing the numbers would
spot a problem.

It is entirely fictitious, and the proof does not require any argument about scores. Ask what it would
take to compute that feature for tomorrow.

In [ ]:
last_day = target.index[-1]
tomorrow = last_day + pd.Timedelta(days=1)

# A centred 7-day window around tomorrow needs the three days after it
window_needed = pd.date_range(tomorrow - pd.Timedelta(days=3), tomorrow + pd.Timedelta(days=3))

print(f"To predict {tomorrow.date()}, the centred 7-day mean needs:")
for day in window_needed:
    available = day <= last_day
    print(f"  {day.date()}  {'known' if available else 'DOES NOT EXIST YET'}")

Three of the seven days the feature needs are in the future. The column can be computed for historical
rows, which is exactly why the mistake survives testing, and it can never be computed for the row you
actually want to predict.

That gives the rule, and it is worth applying to every column you ever add:

> **Every feature on the row for time $t$ must be computable using only data observed strictly before
> $t$.**

Two practical habits follow. Write features as functions of `target.shift(k)` with `k >= 1`, so the shift
is visible in the code rather than implied. And when a result looks surprisingly good, suspect the
features before celebrating.

The `Shuffled K-fold CV` column carries a second, smaller warning. Even with correct features it reports
350 where the time-respecting split reports 481: shuffling rows lets the model train on days surrounding
each test day, which is a milder version of the same error.

**Exercise.** Add `target.shift(-1)` (tomorrow's sales) as a feature and evaluate. The score should be extraordinary. Then check what fraction of the model's predictions that single column accounts for, and convince yourself why no amount of cross-validation would have caught it.

In [ ]:
# Your solution here


---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="6.-Calendar-and-Fourier-Features">6. Calendar and Fourier Features</h3>
</div>

Lags describe what the series has been doing. **Calendar features** describe where in time we are, which
matters whenever behaviour depends on the date itself rather than on recent history.

The naive approach is to extract the parts of the date directly: day of week, month, day of month. Tree
models handle these well, because they can split on them arbitrarily.

The subtlety is that some of these are **cyclical**. Month 12 and month 1 are adjacent in reality and
maximally far apart as integers, and a linear model will believe December and January are opposites.
**Fourier terms** fix this by mapping the cycle onto a circle: a sine and cosine pair per harmonic, which
is the same construction used for dynamic harmonic regression in Notebook
[B03](./B03_Advanced_statistical_models.ipynb).

In [ ]:
def add_calendar(features, index):
    """Where in the calendar each row sits."""
    features["day_of_week"] = index.dayofweek
    features["day_of_month"] = index.day
    features["month"] = index.month
    features["week_of_year"] = index.isocalendar().week.astype(int)
    features["is_weekend"] = (index.dayofweek >= 5).astype(int)
    features["days_since_start"] = (index - index[0]).days   # a linear trend term
    return features


def add_fourier(features, index, period=365.25, harmonics=2):
    """Smooth cyclical position, for a cycle of the given period in days."""
    position = (index.dayofyear if period > 100 else index.dayofweek) / period
    for k in range(1, harmonics + 1):
        features[f"fourier_sin_{k}"] = np.sin(2 * np.pi * k * position)
        features[f"fourier_cos_{k}"] = np.cos(2 * np.pi * k * position)
    return features


demo = pd.DataFrame(index=target.index)
add_calendar(demo, target.index)
add_fourier(demo, target.index)

demo.head(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

year = demo.loc["2014"]
axes[0].plot(year.index, year["month"], color="crimson", linewidth=1.5)
axes[0].set_title("month as an integer", fontsize=13, fontweight="bold")
axes[0].set_ylabel("Value")
axes[0].tick_params(axis="x", rotation=45)

axes[1].plot(year.index, year["fourier_sin_1"], color="steelblue", linewidth=1.5, label="sin")
axes[1].plot(year.index, year["fourier_cos_1"], color="seagreen", linewidth=1.5, label="cos")
axes[1].set_title("the same cycle as Fourier terms", fontsize=13, fontweight="bold")
axes[1].legend()
axes[1].tick_params(axis="x", rotation=45)

for ax in axes:
    ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

The left panel shows the discontinuity: a vertical drop from 12 to 1 at the turn of the year, where
nothing actually happened. The right panel is continuous, and the sine and cosine together identify the
position in the cycle without a seam.

Use both kinds. Tree models are unbothered by the integer version and can carve it up as they like;
linear models and neural networks need the Fourier version. Neither is expensive.

**Known-in-advance columns** belong here too. `Promo`, `SchoolHoliday` and `Open` are not derived from the
past at all: they are facts about the future that the retailer already knows, which makes them exactly the
kind of feature that leakage rules permit and that statistical models struggled to use before Notebook
B02's SARIMAX.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="7.-Preparing-the-Target">7. Preparing the Target</h3>
</div>

So far we have shaped the inputs. The **target** deserves attention too, because a model fits what you
give it, and some shapes are easier to fit than others.

Three questions, each with a test rather than an opinion.

In [ ]:
open_days = target[store["Open"] == 1]

# 1. Is there a unit root?
adf_statistic, adf_p = adfuller(open_days, autolag="AIC")[:2]

# 2. Is there a monotonic trend? Kendall's tau against time, which is the
#    statistic behind the Mann-Kendall trend test
tau, tau_p = stats.kendalltau(np.arange(len(open_days)), open_days.values)

# 3. Is the distribution skewed enough to be worth transforming?
raw_skew = stats.skew(open_days)
log_skew = stats.skew(np.log(open_days))

print(f"ADF test:        statistic={adf_statistic:6.2f}  p={adf_p:.4f}  -> "
      f"{'stationary' if adf_p < 0.05 else 'unit root'}")
print(f"Kendall's tau:   tau={tau:+.3f}  p={tau_p:.4f}  -> "
      f"{'trend present' if tau_p < 0.05 else 'no trend'}")
print(f"Skew:            raw={raw_skew:.2f}   after log={log_skew:.2f}")

The series is stationary by the ADF test, and Kendall's tau finds a **statistically significant but tiny
downward trend**: tau of -0.09 with a p-value of 0.0001. Those two facts are not in conflict, and the
combination is worth pausing on. With 781 observations, a trend far too small to matter commercially is
still easily significant. **Significance is not size.** Add a `days_since_start` column so a model can use
the drift if it helps, and do not detrend the target over a slope this shallow.

The skew is the more actionable result. Sales are right-skewed at 0.93, and a log transform brings that
to 0.25. That matters for models that assume symmetric errors, and it changes what the error metric
rewards: fitting on a log scale penalises proportional errors rather than absolute ones, which for sales
data is often what you want.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)

axes[0].hist(open_days, bins=40, color="steelblue", edgecolor="white")
axes[0].set_title(f"Sales on open days (skew {raw_skew:.2f})", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Sales")
axes[0].set_ylabel("Days")

axes[1].hist(np.log(open_days), bins=40, color="seagreen", edgecolor="white")
axes[1].set_title(f"After a log transform (skew {log_skew:.2f})", fontsize=13, fontweight="bold")
axes[1].set_xlabel("log(Sales)")

for ax in axes:
    ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

> **If you transform, remember to transform back.** A model trained on `log(sales)` predicts logs, and
> `exp()` of the predicted mean is not the mean of the prediction: it is closer to the median, and
> systematically under-forecasts. For most business reporting that bias matters, and it is easy to forget
> because the model's own validation scores, computed on the log scale, look fine.

Differencing is the other standard preparation, and Notebook [B02](./B02_ARIMA_models.ipynb) covered it in
full. It is less common for tree models, which can represent a level shift directly, than for the linear
models and neural networks in Part D.

**Exercise.** Train the model from section 5 on `log(sales)` for open days only, predict, and transform back with `exp`. Compare the MAE against the untransformed model, and check whether the back-transformed forecasts are biased low as the note above predicts.

In [ ]:
# Your solution here


---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="8.-Which-Features-Matter">8. Which Features Matter</h3>
</div>

We now have four families of features: lags, rolling statistics, calendar terms, and known-in-advance
columns. Building them all is cheap. Knowing which ones earn their place is the useful part.

**Permutation importance** answers that directly: shuffle one column in the test data, see how much the
error worsens, and repeat. Unlike the impurity-based importances that tree models report by default, it is
measured on held-out data and on the metric you actually care about, so it cannot reward a feature that
only helps the model memorise the training set.

In [ ]:
def build_full_features(target, store):
    """Everything from this notebook, assembled."""
    features = pd.DataFrame(index=target.index)

    add_lags(features, target, [1, 2, 7, 14, 28])
    add_rolling(features, target, [7, 28])
    add_calendar(features, target.index)
    add_fourier(features, target.index)

    # Known in advance: no shift needed, these are facts about the future
    features["open"] = store["Open"]
    features["promo"] = store["Promo"]
    features["school_holiday"] = store["SchoolHoliday"]

    return features


features = build_full_features(target, store)
complete = features.notna().all(axis=1)
X, y = features[complete], target[complete]

HOLDOUT_DAYS = 90
split = len(X) - HOLDOUT_DAYS

model = HistGradientBoostingRegressor(random_state=0).fit(X.iloc[:split], y.iloc[:split])
predictions = model.predict(X.iloc[split:])

print(f"{X.shape[1]} features, {len(X)} usable rows "
      f"({len(features) - len(X)} lost to the longest lag)")
print(f"Holdout MAE: {mean_absolute_error(y.iloc[split:], predictions):.1f}")

In [ ]:
importance = permutation_importance(
    model, X.iloc[split:], y.iloc[split:],
    n_repeats=15, random_state=0, scoring="neg_mean_absolute_error",
)

ranked = (
    pd.Series(importance.importances_mean, index=X.columns)
    .sort_values(ascending=False)
)

fig, ax = plt.subplots(figsize=(11, 6))

top = ranked.head(14).iloc[::-1]
ax.barh(top.index, top.values, color="steelblue")
ax.set_title("Permutation importance: MAE increase when the column is shuffled",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Increase in MAE")
ax.grid(axis="x", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

ranked.head(8).round(1)

`open` dwarfs everything, which is the right answer and a slightly boring one: a shut shop sells nothing,
and shuffling that column destroys the prediction entirely. `promo` comes second at a distance.

The interesting part is the ranking below those two. **`lag_1` beats every other historical feature**, and
the rolling statistics and longer lags contribute very little on top of it. Once the model knows yesterday,
last week's average adds almost nothing, because yesterday already reflects it.

Read that as a warning against feature-count enthusiasm. We built 24 columns and perhaps six are doing
real work. Extra features are not free: they cost rows at the start of the series, they slow training, and
each one is another chance to introduce leakage.

Notice also how this echoes Notebook B02. There, adding `Open` as an exogenous variable halved SARIMAX's
error and exposed a confounded `Promo` coefficient. Here the same two columns dominate a completely
different model. The structure of the problem, not the choice of algorithm, is what determines which
information matters.

**Exercise.** Drop every feature whose permutation importance is below 10 and refit. How much accuracy do you lose, and how many rows do you get back at the start of the series by no longer needing `lag_28`?

In [ ]:
# Your solution here


---

You can now turn a time series into a feature matrix without letting the future in, and tell which of the
columns are worth keeping.

The next notebook puts models on top of it: linear regression, random forests, and the gradient boosting
libraries, measured against the statistical models of Part B and against the baselines that have embarrassed
them both:
[C02 - Machine Learning Models for Forecasting](./C02_Machine_learning_models.ipynb).